# RQ2 — Dataset Meta-Feature Extraction

**Research Question**: What dataset properties predict which clustering method will generate the best pseudo-labels?

This notebook extracts five meta-feature representations for every dataset:

| Option | Description | Dims | Role |
|--------|-------------|------|------|
| A | Hand-crafted (~30 features via sklearn/scipy) | ~30 | Main approach |
| B | Autoencoder bottleneck (fixed K=4) | 8 | Ablation |
| C | Dictionary Learning sparse codes (fixed K=4) | 8 | Ablation — novel contribution |
| A+B | Option A concatenated with Option B | ~38 | Additive test |
| A+C | Option A concatenated with Option C | ~38 | Main novel claim |

Options B and C use **fixed K=4** so every dataset produces the same-length vector
regardless of n_classes. A sanity check with random features is also included.

> ⚠️  **Fresh-start note**: Delete checkpoint files in `data/meta_table/` if you changed
> the meta-feature code and want a full recomputation.

**Outputs**: `data/meta_table/meta_training_opt{A,B,C,AB,AC}.csv`

In [2]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

ROOT    = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR = os.path.join(ROOT, 'data', 'raw')
META_DIR = os.path.join(ROOT, 'data', 'meta_table')
LSE_CSV  = os.path.join(META_DIR, 'meta_training.csv')
MANIFEST = os.path.join(META_DIR, 'dataset_manifest.csv')

sys.path.insert(0, os.path.join(ROOT, 'src'))
openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)

SHOWCASE_IDS = {61, 187, 15, 53, 40966, 37, 54, 1590, 1597}
print('Paths OK')

Paths OK


In [3]:
lse_df = pd.read_csv(LSE_CSV)
leaked = set(lse_df['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak: {leaked}'

manifest = pd.read_csv(MANIFEST)
n_cls_map = dict(zip(manifest['dataset_id'], manifest['n_classes']))

LSE_COLS    = ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']
DATASET_IDS = lse_df['dataset_id'].tolist()

print(f'Datasets to process: {len(DATASET_IDS)}')

Datasets to process: 86


In [4]:
from metafeatures import extract_optA, extract_optB, extract_optC, extract_optAB, extract_optAC

def load_and_split(dataset_id):
    ds = openml.datasets.get_dataset(
        dataset_id, download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    X = X.select_dtypes(include=[np.number]).astype(float)
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))
    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc, test_size=0.2,
        random_state=SEED, stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te

print('Modules loaded')

Modules loaded


## Option A — Hand-Crafted Meta-Features (~30 features)

In [5]:
CKPT_A = os.path.join(META_DIR, 'mf_checkpoint_A.csv')

if os.path.exists(CKPT_A):
    ckpt_a = pd.read_csv(CKPT_A)
    done_a = set(ckpt_a['dataset_id'])
    rows_a = ckpt_a.to_dict('records')
    print(f'Resuming Option A — {len(done_a)} done')
else:
    done_a, rows_a = set(), []
    print('Starting Option A fresh')

total = len(DATASET_IDS)
for i, did in enumerate(DATASET_IDS):
    if did in done_a:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        feats = extract_optA(X_tr, y_tr, X_te, y_te)
        feats['dataset_id'] = did
        rows_a.append(feats)
        done_a.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_a.append({'dataset_id': did})
        done_a.add(did)
    pd.DataFrame(rows_a).to_csv(CKPT_A, index=False)

optA_df = pd.DataFrame(rows_a)
print(f'\nOption A done. Shape: {optA_df.shape}')

Resuming Option A — 86 done

Option A done. Shape: (86, 31)


## Option B — Autoencoder Bottleneck (fixed K=4, → 8 dims)

In [6]:
CKPT_B = os.path.join(META_DIR, 'mf_checkpoint_B.csv')
K = 4  # fixed bottleneck size — same for all datasets

if os.path.exists(CKPT_B):
    ckpt_b = pd.read_csv(CKPT_B)
    done_b = set(ckpt_b['dataset_id'])
    rows_b = ckpt_b.to_dict('records')
    print(f'Resuming Option B — {len(done_b)} done')
else:
    done_b, rows_b = set(), []
    print('Starting Option B fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_b:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        vec = extract_optB(X_tr, K=K)
        rec = {'dataset_id': did}
        for j, v in enumerate(vec[:K]):
            rec[f'ae_mean_{j}'] = float(v)
        for j, v in enumerate(vec[K:]):
            rec[f'ae_var_{j}'] = float(v)
        rows_b.append(rec)
        done_b.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_b.append({'dataset_id': did})
        done_b.add(did)
    pd.DataFrame(rows_b).to_csv(CKPT_B, index=False)

optB_df = pd.DataFrame(rows_b)
print(f'\nOption B done. Shape: {optB_df.shape}')

Resuming Option B — 86 done

Option B done. Shape: (86, 9)


## Option C — Dictionary Learning Sparse Codes (fixed K=4, → 8 dims)

In [7]:
CKPT_C = os.path.join(META_DIR, 'mf_checkpoint_C.csv')

if os.path.exists(CKPT_C):
    ckpt_c = pd.read_csv(CKPT_C)
    done_c = set(ckpt_c['dataset_id'])
    rows_c = ckpt_c.to_dict('records')
    print(f'Resuming Option C — {len(done_c)} done')
else:
    done_c, rows_c = set(), []
    print('Starting Option C fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_c:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        vec = extract_optC(X_tr, K=K)
        rec = {'dataset_id': did}
        for j, v in enumerate(vec[:K]):
            rec[f'dl_mean_{j}'] = float(v)
        for j, v in enumerate(vec[K:]):
            rec[f'dl_var_{j}'] = float(v)
        rows_c.append(rec)
        done_c.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_c.append({'dataset_id': did})
        done_c.add(did)
    pd.DataFrame(rows_c).to_csv(CKPT_C, index=False)

optC_df = pd.DataFrame(rows_c)
print(f'\nOption C done. Shape: {optC_df.shape}')

Resuming Option C — 86 done

Option C done. Shape: (86, 9)


## Concatenated Variants (A+B and A+C)

These are the honest test of whether autoencoder or dictionary learning features
add signal *beyond* what the hand-crafted features already capture.

In [8]:
for suffix, ckpt_key, extractor in [
    ('AB', 'mf_checkpoint_AB.csv', lambda X_tr, y_tr, X_te, y_te: extract_optAB(X_tr, y_tr, X_te, y_te, K=K)),
    ('AC', 'mf_checkpoint_AC.csv', lambda X_tr, y_tr, X_te, y_te: extract_optAC(X_tr, y_tr, X_te, y_te, K=K)),
]:
    ckpt_path = os.path.join(META_DIR, ckpt_key)
    if os.path.exists(ckpt_path):
        ckpt = pd.read_csv(ckpt_path)
        done_set = set(ckpt['dataset_id'])
        rows = ckpt.to_dict('records')
        print(f'Resuming Option {suffix} — {len(done_set)} done')
    else:
        done_set, rows = set(), []
        print(f'Starting Option {suffix} fresh')

    for i, did in enumerate(DATASET_IDS):
        if did in done_set:
            continue
        t0 = time.time()
        try:
            X_tr, X_te, y_tr, y_te = load_and_split(did)
            feats = extractor(X_tr, y_tr, X_te, y_te)
            feats['dataset_id'] = did
            rows.append(feats)
            done_set.add(did)
            print(f'  [{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
        except Exception as e:
            print(f'  [{i+1:3d}/{total}]  did={did}  FAIL: {e}')
            rows.append({'dataset_id': did})
            done_set.add(did)
        pd.DataFrame(rows).to_csv(ckpt_path, index=False)

    df_concat = pd.DataFrame(rows)
    print(f'Option {suffix} done. Shape: {df_concat.shape}\n')
    # Store for merge step
    if suffix == 'AB':
        optAB_df = df_concat
    else:
        optAC_df = df_concat

Resuming Option AB — 86 done
Option AB done. Shape: (86, 39)

Resuming Option AC — 66 done
  [ 67/86]  did=41825  ok  (26.6s)
  [ 68/86]  did=46840  ok  (95.8s)
  [ 69/86]  did=41731  ok  (8.4s)
  [ 70/86]  did=44791  ok  (3.2s)
  [ 71/86]  did=44626  ok  (41.0s)
  [ 72/86]  did=47052  ok  (0.8s)
  [ 73/86]  did=41709  ok  (30.2s)
  [ 74/86]  did=44719  ok  (91.0s)
  [ 75/86]  did=1075  ok  (2.3s)
  [ 76/86]  did=46880  ok  (45.2s)
  [ 77/86]  did=44702  ok  (155.8s)
  [ 78/86]  did=1459  ok  (130.3s)
  [ 79/86]  did=377  ok  (2.4s)
  [ 80/86]  did=44636  ok  (1.9s)
  [ 81/86]  did=46962  ok  (41.8s)
  [ 82/86]  did=694  ok  (2.7s)
  [ 83/86]  did=4153  ok  (3.0s)
  [ 84/86]  did=941  ok  (0.5s)
  [ 85/86]  did=16  ok  (61.4s)
  [ 86/86]  did=46871  ok  (31.2s)
Option AC done. Shape: (86, 39)



## Sanity Baseline — Random Meta-Features

Random 8-dimensional features (matching Options B and C dimensionality).
If Options B or C cannot beat random in the meta-learner, they are not learning
useful representations — the result would be noise, not a contribution.

In [9]:
rng = np.random.default_rng(SEED)
random_rows = []
for did in DATASET_IDS:
    rec = {'dataset_id': did}
    for j in range(2 * K):
        rec[f'rand_{j}'] = float(rng.normal())
    random_rows.append(rec)

optRand_df = pd.DataFrame(random_rows)
print(f'Random features shape: {optRand_df.shape}')

Random features shape: (86, 9)


## Merge & Save All Tables

In [10]:
TARGET_COLS = ['dataset_id'] + LSE_COLS + ['best_method', 'gt_accuracy']
targets = lse_df[TARGET_COLS]

for letter, mf_df in [
    ('A',    optA_df),
    ('B',    optB_df),
    ('C',    optC_df),
    ('AB',   optAB_df),
    ('AC',   optAC_df),
    ('Rand', optRand_df),
]:
    merged = targets.merge(mf_df, on='dataset_id', how='inner')
    out_path = os.path.join(META_DIR, f'meta_training_opt{letter}.csv')
    merged.to_csv(out_path, index=False)
    leaked = set(merged['dataset_id']) & SHOWCASE_IDS
    assert len(leaked) == 0, f'Showcase leak in opt{letter}'
    print(f'Option {letter} saved → {out_path}  shape={merged.shape}')

# Update main meta_training.csv with Option A features
main_df = lse_df.merge(optA_df, on='dataset_id', how='left')
main_df.to_csv(LSE_CSV, index=False)
print(f'Main table updated → {LSE_CSV}  shape={main_df.shape}')

Option A saved → c:\MLResearch\data\meta_table\meta_training_optA.csv  shape=(86, 39)
Option B saved → c:\MLResearch\data\meta_table\meta_training_optB.csv  shape=(86, 17)
Option C saved → c:\MLResearch\data\meta_table\meta_training_optC.csv  shape=(86, 17)
Option AB saved → c:\MLResearch\data\meta_table\meta_training_optAB.csv  shape=(86, 47)
Option AC saved → c:\MLResearch\data\meta_table\meta_training_optAC.csv  shape=(86, 47)
Option Rand saved → c:\MLResearch\data\meta_table\meta_training_optRand.csv  shape=(86, 17)
Main table updated → c:\MLResearch\data\meta_table\meta_training.csv  shape=(86, 39)


In [11]:
df_a = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
mf_cols = [c for c in df_a.columns if c not in TARGET_COLS]

print(f'=== Option A: {len(mf_cols)} meta-features ===')
print('Features:', mf_cols)

nan_counts = df_a[mf_cols].isna().sum()
if nan_counts.any():
    print('\nNaN counts:')
    print(nan_counts[nan_counts > 0])
else:
    print('\nNo NaN values in Option A.')

print('\nAll sanity checks passed.')
print('Ready for 05_meta_learner.ipynb')

=== Option A: 30 meta-features ===
Features: ['n_instances', 'n_features', 'n_classes', 'skewness_mean', 'kurtosis_mean', 'mean_abs_pearson', 'class_entropy', 'imbalance_ratio', 'hopkins', 'intrinsic_dim_ratio', 'pca_var_pc1', 'pca_top3_var', 'pca_entropy', 'inter_intra_ratio', 'pairwise_dist_mean', 'pairwise_dist_cv', 'pairwise_dist_p90', 'knn5_dist_mean', 'knn5_dist_cv', 'knn5_dist_p90', 'silhouette_true', 'davies_bouldin_true', 'knn1_accuracy', 'decision_stump_accuracy', 'feature_sparsity', 'high_corr_frac', 'corr_dispersion', 'feature_std_dispersion', 'zero_variance_frac', 'cv_mean']

NaN counts:
skewness_mean       3
kurtosis_mean       3
mean_abs_pearson    3
dtype: int64

All sanity checks passed.
Ready for 05_meta_learner.ipynb
